# Forelesning 3: Deskriptiv statistikk i kjemi

Når vi gjentar en kjemisk måling, får vi sjelden nøyaktig samme resultat. I dag skal vi svare på tre spørsmål:

1. **Hvor ligger målingene?** Gjennomsnitt, median og persentiler
2. **Hvor mye varierer de?** Variasjonsbredde, interkvartilbredde, standardavvik og RSD
3. **Hvor sikkert kjenner vi gjennomsnittet?** Standardfeil og konfidensintervall

Målet er ikke å pugge formler, men å kunne velge riktig størrelse og tolke den kjemisk.

Menti: [lenke]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t

plt.rcParams.update({"font.size": 14, "figure.figsize": (8, 4.5)})

## Del 1: Hvorfor varierer målinger?

Vi har målt jernkonsentrasjonen i den samme løsningen fem ganger:

```python
iron_mg_L = [5.12, 5.08, 5.15, 5.10, 5.13]
```

<div class="viktig">

Hva slags data vi har, bestemmer hvilken statistikk som passer:

- **Gjentatte målinger av samme prøve:** ofte gjennomsnitt og standardavvik
- **Målinger fra mange ulike prøver**, for eksempel nitrat i mange bekker: fordelingen er ofte skjev, og median og interkvartilbredde er ofte mer informative

</div>

<div class="diskuter">

**Diskuter:** Hva kan være årsaken til at målingene ikke er like?

</div>

## Del 2: Hvor ligger målingene?

**Gjennomsnittet** bruker alle målingene. **Medianen** er den midterste verdien når målingene er sortert.

In [ ]:
iron_mg_L = np.array([5.12, 5.08, 5.15, 5.10, 5.13])

mean = np.mean(iron_mg_L)
median = np.median(iron_mg_L)

print(f"Mean: {mean:.3f} mg/L")
print(f"Median: {median:.3f} mg/L")

<div class="oppgave">

<div class="bilde-hoyre" style="width:24%">

![Pipettering](bilder/pipette.jpg)
<p class="kreditering">Foto: [kilde og lisens]</p>

</div>

**Prøv selv (Menti):** I serie 2 har den siste målingen blitt 10,42 i stedet for 10,04, for eksempel på grunn av en feilpipettering. **Tipp først:** Hvilken størrelse endres mest, gjennomsnittet eller medianen? Kjør deretter koden.

</div>

In [ ]:
series_1 = np.array([10.02, 10.05, 9.98, 10.01, 10.04])
series_2 = np.array([10.02, 10.05, 9.98, 10.01, 10.42])

for series in [series_1, series_2]:
    print(f"Mean: {np.mean(series):.3f}   Median: {np.median(series):.3f}")

<div class="viktig">

Medianen er **robust**: den påvirkes lite av enkeltverdier som er svært høye eller lave. Gjennomsnittet bruker all informasjonen, men én avvikende måling kan flytte det merkbart.

</div>

<div class="diskuter">

**Diskuter:** Betyr dette at vi bare kan fjerne 10,42? Hva må vi gjøre først?

</div>

## Del 3: Persentiler og boksplott

<div class="viktig">

En **persentil** er en grenseverdi i et sortert datasett. 25-persentilen er verdien som omtrent 25 % av observasjonene ligger på eller under. 50-persentilen er medianen. En persentil er **ikke** en prosent av den største verdien.

</div>

<div class="bilde-hoyre" style="width:30%">

![Vannprøvetaking i bekk](bilder/bekk_provetaking.jpg)
<p class="kreditering">Foto: [kilde og lisens]</p>

</div>

Nitratkonsentrasjonen er målt i tolv bekker. De fleste har lave verdier, men noen bekker ligger nær jordbruksområder.

In [ ]:
nitrate_mg_L = np.array([0.42, 0.47, 0.50, 0.55, 0.58, 0.61,
                         0.65, 0.72, 0.85, 1.10, 1.35, 2.80])

q1 = np.percentile(nitrate_mg_L, 25)
median = np.percentile(nitrate_mg_L, 50)
q3 = np.percentile(nitrate_mg_L, 75)

print(f"Q1: {q1:.2f} mg/L   Median: {median:.2f} mg/L   Q3: {q3:.2f} mg/L")
print(f"Mean: {np.mean(nitrate_mg_L):.2f} mg/L")

<div class="diskuter">

**Diskuter:** Hva betyr det at en bekk ligger over 75-persentilen? Hvorfor er gjennomsnittet høyere enn medianen her?

</div>

### Boksplott
Et boksplott viser de samme størrelsene grafisk.

In [ ]:
plt.boxplot(nitrate_mg_L, vert=False)
plt.xlabel("Nitrate concentration (mg/L)")
plt.title("Nitrate in stream-water samples")
plt.show()

<div class="viktig">

<div class="bilde-midt" style="width:90%">

![Annotert boksplott](bilder/boksplott_forklart.png)

</div>

Boksen inneholder de midterste 50 % av målingene. Faktoren 1,5 for værhårene er en konvensjon, og punkter utenfor må undersøkes før vi eventuelt fjerner dem.

</div>

## Del 4: Hvor mye varierer dataene?

| Størrelse | Definisjon | Påvirkes av ekstreme verdier? |
|---|---|---|
| Variasjonsbredde | største minus minste verdi | svært mye |
| Interkvartilbredde (IQR) | $Q_3 - Q_1$ | lite |
| Standardavvik ($s$) | typisk avstand fra gjennomsnittet | ja |

In [ ]:
data_range = np.max(nitrate_mg_L) - np.min(nitrate_mg_L)
iqr = q3 - q1

print(f"Range: {data_range:.2f} mg/L")
print(f"IQR: {iqr:.2f} mg/L")

### Standardavvik

$$s=\sqrt{\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}}$$

Vi deler på $n-1$ fordi målingene er et utvalg. I NumPy må vi be om det med `ddof=1`. Uten `ddof=1` deler NumPy på $n$.

In [ ]:
sd = np.std(iron_mg_L, ddof=1)
print(f"SD: {sd:.3f} mg/L")

<div class="viktig">

Standardavviket har **samme enhet** som målingene og beskriver den typiske spredningen til **enkeltmålingene** rundt gjennomsnittet. Det betyr ikke at alle målingene ligger nøyaktig så langt fra gjennomsnittet.

</div>

### Relativt standardavvik (RSD)

$$RSD=\frac{s}{\bar{x}}\cdot 100\ \%$$

RSD setter spredningen i forhold til målenivået, og brukes mye for å sammenlikne presisjonen til analysemetoder.

<div class="oppgave">

**Prøv selv:** Skriv en funksjon `rsd(measurements)` som returnerer RSD i prosent. Test den på jernmålingene.

</div>

In [ ]:
def rsd(measurements):
    """Relativt standardavvik i prosent."""
    return np.std(measurements, ddof=1) / np.mean(measurements) * 100

print(f"RSD: {rsd(iron_mg_L):.2f} %")

<div class="diskuter">

**Diskuter:** Metode A gir 1,0 mg/L med standardavvik 0,05 mg/L. Metode B gir 100 mg/L med standardavvik 0,20 mg/L. Hvilken metode har minst spredning, og hvilken har best relativ presisjon? Når blir RSD misvisende?

</div>

## Hva bruker vi når?

<div class="viktig">

| Situasjon | Nyttige størrelser |
|---|---|
| Gjentatte målinger av samme prøve, omtrent symmetrisk | gjennomsnitt og standardavvik |
| Mange ulike prøver med skjev fordeling | median og IQR |
| Sammenlikne presisjon ved ulike konsentrasjoner | RSD |
| Hvor sikkert kjenner vi gjennomsnittet? | standardfeil eller konfidensintervall |

Dette er gode utgangspunkt, ikke automatiske regler.

</div>

<div class="diskuter">

**Menti:** Hva ville du brukt for 1) seks bestemmelser av klorid i samme mineralvann, 2) nitrat i 80 brønner der noen få verdier er svært høye, 3) bly i jord fra ulike steder i et gammelt industriområde?

</div>

## Del 5: Fra spredning til usikkerhet i gjennomsnittet

Hvis vi gjentar **hele forsøket** med fem nye målinger, får vi et litt annet gjennomsnitt. Hvor mye varierer gjennomsnittet?

Det kan vi undersøke med en simulering. Vi later som vi kjenner den sanne konsentrasjonen og spredningen, og trekker tilfeldige målinger med `rng.normal(gjennomsnitt, standardavvik, antall)`.

- Hva gjør programmet nedenfor?
- Hvorfor er histogrammet smalere enn spredningen til enkeltmålingene?

In [ ]:
rng = np.random.default_rng(42)

true_mean = 5.12   # mg/L, i virkeligheten kjenner vi ikke denne
true_sd = 0.027    # mg/L
n = 5

means = []
for i in range(1000):
    experiment = rng.normal(true_mean, true_sd, n)
    means.append(np.mean(experiment))

print(f"SD of single measurements: {true_sd:.4f} mg/L")
print(f"SD of the 1000 means:      {np.std(means, ddof=1):.4f} mg/L")
print(f"s / sqrt(n):               {true_sd / np.sqrt(n):.4f} mg/L")

plt.hist(means, bins=30)
plt.xlabel("Mean of n measurements (mg/L)")
plt.ylabel("Number of experiments")
plt.show()

<div class="viktig">

**Standardfeilen** beskriver hvor mye gjennomsnittet varierer fra forsøk til forsøk:

$$SE=\frac{s}{\sqrt{n}}$$

Standardavviket beskriver **enkeltmålingene**. Standardfeilen beskriver **gjennomsnittet**.

</div>

<div class="oppgave">

**Prøv selv:** Endre `n` til 20 og kjør simuleringen på nytt. Hva skjer med histogrammet? Hvor mange målinger trenger du for å halvere standardfeilen?

</div>

## Del 6: Konfidensintervall

<div class="viktig">

Et **95 % konfidensintervall** lages med en metode som, dersom hele forsøket gjentas mange ganger, gir intervaller som inneholder det sanne gjennomsnittet omtrent 95 % av gangene.

$$\bar{x}\pm t\cdot\frac{s}{\sqrt{n}}$$

</div>

Vi kjenner ikke det sanne standardavviket, bare anslaget $s$. Derfor bruker vi en **t-faktor** fra Students t-fordeling. Den avhenger av antall frihetsgrader, $df = n-1$.

In [ ]:
for df in [1, 2, 4, 9, 29, 1000]:
    print(f"df = {df:4d}   t = {t.ppf(0.975, df):.3f}")

<div class="diskuter">

**Diskuter:** Hvorfor er t-faktoren så stor når vi har få målinger? Hva nærmer den seg når $n$ blir stort?

</div>

### Konfidensintervall for jernmålingene

In [ ]:
n = len(iron_mg_L)
mean = np.mean(iron_mg_L)
sd = np.std(iron_mg_L, ddof=1)
se = sd / np.sqrt(n)

t_factor = t.ppf(0.975, n - 1)   # 97,5 % fordi intervallet er tosidig
margin = t_factor * se

print(f"Mean: {mean:.3f} mg/L")
print(f"95 % CI: [{mean - margin:.3f}, {mean + margin:.3f}] mg/L")

Vi samler stegene i en funksjon, slik at vi kan bruke den på nytt.

In [ ]:
def confidence_interval(measurements, level=0.95):
    """Returnerer nedre og øvre grense for konfidensintervallet til gjennomsnittet."""
    n = len(measurements)
    mean = np.mean(measurements)
    se = np.std(measurements, ddof=1) / np.sqrt(n)
    t_factor = t.ppf(1 - (1 - level) / 2, n - 1)
    return mean - t_factor * se, mean + t_factor * se

lower, upper = confidence_interval(iron_mg_L)
print(f"95 % CI: [{lower:.3f}, {upper:.3f}] mg/L")

### Hva betyr 95 %?
Vi gjentar hele forsøket 100 ganger i en simulering og lager et intervall hver gang.

<div class="oppgave">

**Prøv selv:** Fyll inn vilkåret som sjekker om intervallet inneholder det sanne gjennomsnittet. Hvor mange av de 100 intervallene treffer?

</div>

In [ ]:
hits = 0
for i in range(100):
    experiment = rng.normal(true_mean, true_sd, 5)
    lower, upper = confidence_interval(experiment)

    if lower <= true_mean and true_mean <= upper:
        hits += 1
        color = "tab:blue"
    else:
        color = "tab:red"
    plt.plot([lower, upper], [i, i], color=color)

plt.axvline(true_mean, color="black")
plt.xlabel("Iron concentration (mg/L)")
plt.ylabel("Experiment number")
plt.show()

print(f"{hits} of 100 intervals contain the true mean")

<div class="viktig">

Det er **metoden** som treffer omtrent 95 % av gangene. Et konfidensintervall er **ikke**

- området der 95 % av enkeltmålingene ligger
- en garanti mot systematiske feil: Er standardløsningen feil, kan vi få et smalt intervall rundt et gjennomsnitt som er feil

</div>

<div class="diskuter">

<div class="bilde-hoyre" style="width:26%">

![Kyvetter](bilder/kyvetter.jpg)
<p class="kreditering">Foto: [kilde og lisens]</p>

</div>

**Diskuter:** Hva gjør intervallet smalere? Hvorfor sier ti avlesninger av samme kyvette mindre om analysemetoden enn ti uavhengig preparerte løsninger?

</div>

## Rapportering og avrunding

<div class="viktig">

Oppgi usikkerheten med **maks to gjeldende sifre**, og avrund gjennomsnittet til **samme desimalplass** som usikkerheten. Behold full presisjon i beregningene, og avrund først når resultatet skal presenteres. Oppgi alltid hva usikkerheten er (SD, SE eller 95 % KI).

</div>

| Python gir | Rapporter |
|---|---|
| 5,1164 ± 0,02736 mg/L | 5,116 ± 0,027 mg/L |
| 5,1164 ± 0,03184 mg/L | 5,12 ± 0,03 mg/L |

<div class="diskuter">

**Menti:** Hvordan rapporterer du 0,60433 ± 0,003786 og 12,4150 ± 0,0271?

</div>

## Del 7: Feilstolper

Magnesium er målt ved fem konsentrasjoner, med tre replikater ved hver konsentrasjon. Radene er konsentrasjoner, og kolonnene er replikater. `axis=1` beregner langs hver rad.

In [ ]:
concentration_mg_L = np.array([0.2, 0.3, 0.4, 0.5, 0.6])
absorbance = np.array([
    [0.21, 0.22, 0.19],
    [0.26, 0.29, 0.24],
    [0.33, 0.33, 0.34],
    [0.41, 0.42, 0.45],
    [0.56, 0.61, 0.58],
])

mean_abs = np.mean(absorbance, axis=1)
sd_abs = np.std(absorbance, axis=1, ddof=1)

plt.errorbar(concentration_mg_L, mean_abs, yerr=sd_abs, fmt="o", capsize=4)
plt.xlabel("Mg concentration (mg/L)")
plt.ylabel("Absorbance")
plt.title("Error bars: SD (n = 3)")
plt.show()

<div class="oppgave">

**Prøv selv:** Beregn standardfeilen og feilmarginen til 95 % konfidensintervallet for hver konsentrasjon. Programmet tegner så de tre typene feilstolper ved siden av hverandre.

</div>

In [ ]:
n = 3
se_abs = sd_abs / np.sqrt(n)
ci_abs = t.ppf(0.975, n - 1) * se_abs

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, errors, name in zip(axes, [sd_abs, se_abs, ci_abs], ["SD", "SE", "95 % CI"]):
    ax.errorbar(concentration_mg_L, mean_abs, yerr=errors, fmt="o", capsize=4)
    ax.set_title(name)
    ax.set_xlabel("Mg concentration (mg/L)")
axes[0].set_ylabel("Absorbance")
plt.show()

<div class="diskuter">

**Diskuter:** Med tre replikater blir konfidensintervallet *lengre* enn standardavviket. Hvorfor? Hvorfor må figurteksten alltid si hva feilstolpene viser?

</div>

## Del 8: Statistikk med Pandas

Pandas har de samme metodene: `mean()`, `median()`, `std()`, `quantile()` og `describe()`. `std()` i Pandas deler på $n-1$ uten at vi må be om det. Vi bruker UV-Vis-dataene fra forrige forelesning.

In [ ]:
import pandas as pd

calibration = pd.read_csv("../datahandtering_visualisering/data/uvvis_calibration.csv")
standards = calibration[calibration["sample_type"] == "standard"]

summary = standards.groupby("concentration_uM")["absorbance"].agg(["count", "mean", "std"])
summary["rsd_percent"] = summary["std"] / summary["mean"] * 100
summary

<div class="oppgave">

**Prøv selv:** Den ukjente prøven er målt tre ganger. Hent ut absorbansene, og beregn gjennomsnitt, standardavvik, RSD og 95 % konfidensintervall med funksjonene vi har skrevet. Rapporter resultatet med riktig avrunding.

</div>

In [ ]:
unknown = calibration[calibration["sample_type"] == "unknown"]["absorbance"]

print(unknown.describe())
print(f"RSD: {rsd(unknown):.2f} %")
lower, upper = confidence_interval(unknown)
print(f"95 % CI: [{lower:.4f}, {upper:.4f}]")

<div class="diskuter">

**Diskuter:** Hva forteller standardavviket, og hva forteller konfidensintervallet om den ukjente prøven? Hva sier de *ikke* noe om?

</div>

__Oppgaver__:
1. Kobberkonsentrasjonen i åtte jordprøver er `[12, 14, 15, 16, 18, 21, 29, 47]` mg/kg. Beregn 25-, 50- og 75-persentilen, og forklar hvert resultat med ord. Lag et boksplott.
2. Beregn gjennomsnitt, median, variasjonsbredde og IQR for `[4.91, 5.03, 4.98, 5.01, 5.07]`. Legg til `5.80` og beregn på nytt. Hvilke størrelser påvirkes mest, og hvorfor?
3. Beregn gjennomsnitt, standardavvik, standardfeil og 95 % konfidensintervall for `[12.42, 12.38, 12.45, 12.41, 12.39, 12.44]` mg/L. Rapporter resultatet med riktig avrunding, og forklar to endringer i forsøket som kan gjøre intervallet smalere.
4. En student måler samme løsning ti ganger uten å ta kyvetten ut av instrumentet. En annen lager ti løsninger uavhengig av hverandre og måler hver én gang. Hvilke kilder til variasjon fanger de to forsøkene opp? Hvilket forsøk sier mest om presisjonen til analysemetoden?
5. Bruk simuleringen av konfidensintervaller og endre konfidensnivået til 90 % og 99 %. Hvor mange intervaller treffer nå, og hva skjer med bredden?